# Plant Disease Detection - Hybrid Quantum-Classical Model

This notebook demonstrates training a hybrid quantum-classical model for plant disease classification.

## 1. Setup and Imports

In [ ]:
import time
import os
import copy
import json
import csv
from pathlib import Path

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import datasets, transforms
import torch.nn.functional as F

# Pennylane
import pennylane as qml
from pennylane import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics
from sklearn.metrics import confusion_matrix

from metrics_utils import count_trainable_parameters, evaluate_model

# Set seeds for reproducibility
GLOBAL_SEED = 42
torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# OpenMP: number of parallel threads
os.environ["OMP_NUM_THREADS"] = "1"


## 2. Configuration

In [ ]:
# Hyperparameters
n_qubits = 4                # Number of qubits
step = 0.0004              # Learning rate
batch_size = 16             # Batch size
num_epochs = 10             # Number of training epochs
n_classes = 4               # Number of classes
q_depth = 4                 # Depth of quantum circuit
gamma_lr_scheduler = 0.1    # Learning rate reduction factor
q_delta = 0.01              # Initial spread of quantum weights

# Data paths
data_dir = 'data'           # Root data directory
processed_dir = 'scl_metadata'  # Processed train/val split directory

# Device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 3. Data Preparation

In [ ]:
import shutil
import random

# Create train/val split if it doesn't exist
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)
    train_dir = os.path.join(processed_dir, 'train')
    val_dir = os.path.join(processed_dir, 'val')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)
    
    total_images_per_class = 400
    train_ratio = 0.8
    
    # Process each class
    for class_name in sorted(os.listdir(data_dir)):
        class_dir = os.path.join(data_dir, class_name)
        if os.path.isdir(class_dir):
            os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
            os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)
            
            images = [os.path.join(class_dir, f) for f in os.listdir(class_dir) 
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.JPG'))]
            
            random.shuffle(images)
            images = images[:total_images_per_class]
            
            num_train = int(len(images) * train_ratio)
            train_images = images[:num_train]
            val_images = images[num_train:]
            
            for img in train_images:
                shutil.copy(img, os.path.join(train_dir, class_name, os.path.basename(img)))
            for img in val_images:
                shutil.copy(img, os.path.join(val_dir, class_name, os.path.basename(img)))
    
    print(f"Dataset created with {total_images_per_class} images per class")
else:
    print("Processed dataset already exists")


In [ ]:
# Data transforms
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    "val": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

# Load datasets
image_datasets = {
    x if x == "train" else "validation": datasets.ImageFolder(
        os.path.join(processed_dir, x), data_transforms[x]
    )
    for x in ["train", "val"]
}

dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "validation"]}
class_names = image_datasets["train"].classes

print(f"Classes: {class_names}")
print(f"Dataset sizes: {dataset_sizes}")

# Create dataloaders
dataloaders = {
    x: torch.utils.data.DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True)
    for x in ["train", "validation"]
}


## 4. Quantum Circuit Definition


In [ ]:
# Initialize quantum device
dev = qml.device("default.qubit", wires=n_qubits)

def H_layer(nqubits):
    """Layer of single-qubit Hadamard gates."""
    for idx in range(nqubits):
        qml.Hadamard(wires=idx)

def RY_layer(w):
    """Layer of parametrized qubit rotations around the y axis."""
    for idx, element in enumerate(w):
        qml.RY(element, wires=idx)

def entangling_layer(nqubits):
    """Layer of CNOTs followed by another shifted layer of CNOT."""
    for i in range(0, nqubits - 1, 2):
        qml.CNOT(wires=[i, i + 1])
    for i in range(1, nqubits - 1, 2):
        qml.CNOT(wires=[i, i + 1])

@qml.qnode(dev)
def quantum_net(q_input_features, q_weights_flat):
    """The variational quantum circuit."""
    q_weights = q_weights_flat.reshape(q_depth, n_qubits)
    H_layer(n_qubits)
    RY_layer(q_input_features)
    for k in range(q_depth):
        entangling_layer(n_qubits)
        RY_layer(q_weights[k])
    exp_vals = [qml.expval(qml.PauliZ(position)) for position in range(n_qubits)]
    return tuple(exp_vals)


## 5. Model Definition

In [ ]:
class DressedQuantumNet(nn.Module):
    """Dressed quantum network."""
    
    def __init__(self):
        super().__init__()
        self.pre_net = nn.Linear(512, n_qubits)
        self.q_params = nn.Parameter(q_delta * torch.randn(q_depth * n_qubits))
        self.post_net = nn.Linear(n_qubits, n_classes)
    
    def forward(self, input_features):
        pre_out = self.pre_net(input_features)
        q_in = torch.tanh(pre_out) * np.pi / 2.0
        
        q_out = torch.Tensor(0, n_qubits).to(input_features.device)
        for elem in q_in:
            q_out_elem = torch.hstack(quantum_net(elem, self.q_params)).float().unsqueeze(0)
            q_out = torch.cat((q_out, q_out_elem))
        
        return self.post_net(q_out)

# Create hybrid model
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
model_hybrid = torchvision.models.resnet18(weights=weights)

# Freeze backbone
for param in model_hybrid.parameters():
    param.requires_grad = False

# Replace final layer with quantum circuit
model_hybrid.fc = DressedQuantumNet()
model_hybrid = model_hybrid.to(device)

print("Model created successfully")


## 6. Training Setup


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer_hybrid = optim.Adam(model_hybrid.fc.parameters(), lr=step)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_hybrid, step_size=10, gamma=gamma_lr_scheduler)

print("Training setup complete")


## 7. Training Loop


In [ ]:
def train_model(model, dataloaders, dataset_sizes, criterion, optimizer, scheduler, num_epochs, device):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    for epoch in range(num_epochs):
        for phase in ["train", "validation"]:
            if phase == "train":
                model.train()
            else:
                model.eval()
            
            running_loss = 0.0
            running_corrects = 0
            
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data).item()
            
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects / dataset_sizes[phase]
            
            if phase == "train":
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc)
                scheduler.step()
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
            
            print(f"Phase: {phase:12s} Epoch: {epoch+1}/{num_epochs} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
    
    model.load_state_dict(best_model_wts)
    time_elapsed = time.time() - since
    print(f"\nTraining completed in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best validation accuracy: {best_acc:.4f}")
    
    return model, history

# Train the model
model_hybrid, history = train_model(
    model_hybrid, dataloaders, dataset_sizes, criterion, 
    optimizer_hybrid, exp_lr_scheduler, num_epochs, device
)


## 8. Evaluation


In [ ]:
# Evaluate
results = evaluate_model(model_hybrid, dataloaders['validation'], device, return_confusion_matrix=True)

print("\nEvaluation Results:")
print(f"Accuracy:  {results['accuracy']:.4f}")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall:    {results['recall']:.4f}")
print(f"F1 Score:  {results['f1_score']:.4f}")


## 8b. Frozen Linear Head Baseline

Parameter-matched classical control: frozen ResNet18 + `Linear(512 → K)` trained with the same protocol as the hybrid head.

In [ ]:
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
model_linear = torchvision.models.resnet18(weights=weights)

for param in model_linear.parameters():
    param.requires_grad = False

model_linear.fc = nn.Linear(512, n_classes)
model_linear = model_linear.to(device)

print(f"Linear head trainable parameters: {count_trainable_parameters(model_linear):,}")
print(f"Hybrid head trainable parameters: {count_trainable_parameters(model_hybrid):,}")

In [ ]:
optimizer_linear = optim.Adam(model_linear.fc.parameters(), lr=step)
scheduler_linear = lr_scheduler.StepLR(optimizer_linear, step_size=10, gamma=gamma_lr_scheduler)

model_linear, history_linear = train_model(
    model_linear, dataloaders, dataset_sizes, criterion,
    optimizer_linear, scheduler_linear, num_epochs, device
)

In [ ]:
results_linear = evaluate_model(model_linear, dataloaders['validation'], device)

print("\nLinear Head Evaluation:")
print(f"Accuracy:  {results_linear['accuracy']:.4f}")
print(f"Precision: {results_linear['precision']:.4f}")
print(f"Recall:    {results_linear['recall']:.4f}")
print(f"F1 Score:  {results_linear['f1_score']:.4f}")

# Merge with existing S1 baselines and save elevation CSV
baseline_csv = Path('../results/tables/s1_four_unrelated.csv')
elevation_csv = Path('../../results/elevation/s1_linear_head.csv')
elevation_csv.parent.mkdir(parents=True, exist_ok=True)

baselines = {}
with baseline_csv.open(newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        baselines[row['metric']] = row

metric_keys = ('accuracy', 'precision', 'recall', 'f1_score')
with elevation_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=['metric', 'linear_head', 'hybrid', 'resnet18_ft', 'densenet121_ft', 'simple_cnn'],
    )
    writer.writeheader()
    for metric in metric_keys:
        base = baselines.get(metric, {})
        writer.writerow({
            'metric': metric,
            'linear_head': f"{results_linear[metric]:.2f}",
            'hybrid': base.get('hybrid', ''),
            'resnet18_ft': base.get('resnet18', ''),
            'densenet121_ft': base.get('densenet121', ''),
            'simple_cnn': base.get('simple_cnn', ''),
        })

print(f"\nElevation CSV saved to {elevation_csv.resolve()}")

## 9. Visualization


In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

epochs = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
ax1.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
ax1.set_title('Training and Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, history['train_acc'], 'b-', label='Train Accuracy')
ax2.plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    results['confusion_matrix'], 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names, 
    yticklabels=class_names
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()


## 10. Save Model


In [ ]:
# Save model
os.makedirs('torch_saved', exist_ok=True)
torch.save(model_hybrid.state_dict(), 'torch_saved/model_hybrid.tch')
print("Model saved to torch_saved/model_hybrid.tch")

# Save training history
with open('training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print("Training history saved to training_history.json")
